<a href="https://colab.research.google.com/github/Karna14314/Nanogpt/blob/main/Nanogpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 nanoGPT TinyStories — Train & Deploy to Hugging Face
This notebook implements a compute-efficient character-level transformer trained on TinyStories.

**Pipeline:**
1. Clone nanoGPT & install deps
2. Download & prepare TinyStories dataset
3. Configure training for Colab T4 GPU
4. Train the model
5. Generate samples
6. Export model + tokenizer to Hugging Face Hub
7. Deploy a Gradio demo to Hugging Face Spaces

In [ ]:
# Step 1: Clone nanoGPT and install dependencies
import os
if not os.path.exists('nanoGPT'):
    !git clone https://github.com/karpathy/nanoGPT.git
%cd nanoGPT
!pip install -q tiktoken requests numpy torch huggingface_hub transformers gradio

### 2. Dataset — TinyStories
High linguistic signal allows a small model to learn grammar and story structure faster than Wikipedia.

In [ ]:
import os, requests

os.makedirs('data/tinystories', exist_ok=True)

data_url = 'https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-train.txt'
input_file_path = 'data/tinystories/input.txt'

if not os.path.exists(input_file_path):
    print("Downloading TinyStories subset (~20 MB)...")
    r = requests.get(data_url, stream=True)
    with open(input_file_path, 'w', encoding='utf-8') as f:
        count = 0
        for chunk in r.iter_content(chunk_size=1024*1024):
            f.write(chunk.decode('utf-8', errors='ignore'))
            count += 1
            if count >= 20:
                break
    print("Download complete.")
else:
    print("Dataset already exists, skipping download.")

print(f"File size: {os.path.getsize(input_file_path) / 1024 / 1024:.1f} MB")

In [ ]:
import numpy as np
import pickle

# Read data
with open(input_file_path, 'r', encoding='utf-8') as f:
    data = f.read()

# Character-level tokenizer
chars = sorted(list(set(data)))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

# Save metadata
meta = {'vocab_size': vocab_size, 'itos': itos, 'stoi': stoi}
with open('data/tinystories/meta.pkl', 'wb') as f:
    pickle.dump(meta, f)

# Encode & split 90/10
encode = lambda s: [stoi[c] for c in s]
n = int(len(data) * 0.9)
train_ids = np.array(encode(data[:n]), dtype=np.uint16)
val_ids   = np.array(encode(data[n:]), dtype=np.uint16)

train_ids.tofile('data/tinystories/train.bin')
val_ids.tofile('data/tinystories/val.bin')

print(f"Vocab size: {vocab_size}")
print(f"Train tokens: {len(train_ids):,}  |  Val tokens: {len(val_ids):,}")
print(f"Saved train.bin, val.bin, and meta.pkl")

### 3. Training Configuration
Optimized for Colab T4 GPU:
- **`dtype='float16'`** — T4 does NOT support bfloat16 natively
- **`compile=False`** — `torch.compile` is very slow on T4 and throws warnings; disabling it gives faster wall-clock time for short runs
- **`wandb_log=False`** — no W&B overhead
- **`always_save_checkpoint=True`** — ensures we get final weights

In [ ]:
config_text = """
# ── nanoGPT TinyStories config (Colab T4) ──
out_dir = 'out-tinystories'
eval_interval = 100
eval_iters = 20
log_interval = 10

# Ensure checkpoint is always saved at end of training
always_save_checkpoint = True

# No W&B logging
wandb_log = False

dataset = 'tinystories'
batch_size = 64
block_size = 256

init_from = 'scratch'

# Compact model: ~1.6M params
n_layer = 6
n_head = 6
n_embd = 192
dropout = 0.1

learning_rate = 1e-3
max_iters = 2000
lr_decay_iters = 2000
min_lr = 1e-4
beta2 = 0.99
warmup_iters = 100

# ── T4-critical settings ──
device = 'cuda'
dtype = 'float16'        # T4 does NOT support bfloat16
compile = False          # torch.compile is too slow on T4 for short runs
"""

os.makedirs('config', exist_ok=True)
with open('config/train_tinystories.py', 'w') as f:
    f.write(config_text)

print("✅ Training config written to config/train_tinystories.py")

### 4. Train the Model
Running 2000 iterations — should take ~5-10 min on a T4 without `torch.compile`.

In [ ]:
!python train.py config/train_tinystories.py

### 5. Sample Generation
Generate text to check the model learned coherent narrative patterns.

In [ ]:
!python sample.py --out_dir=out-tinystories --device=cuda --dtype=float16 --compile=False --num_samples=3 --max_new_tokens=200

---
## 6. Export to Hugging Face Hub 🤗
We will:
1. Load the checkpoint
2. Convert it into a standalone PyTorch model + custom tokenizer
3. Push everything (model weights, tokenizer, config, model card) to Hugging Face

In [ ]:
# ── Authenticate with Hugging Face ──
import os
from huggingface_hub import HfApi, login

# Securely load the HF token
hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    print("✅ Loaded HF token from Colab Secrets")
except Exception:
    # If not in Colab secrets, check environment variables
    hf_token = os.environ.get('HF_TOKEN')

if not hf_token:
    # If no token is found, prompt the user manually instead of hardcoding
    from getpass import getpass
    print("HF_TOKEN not found in Secrets. Please paste your token below:")
    hf_token = getpass("Hugging Face Token: ")

login(token=hf_token)
api = HfApi()
whoami = api.whoami()
HF_USERNAME = whoami['name']
print(f"✅ Logged in as: {HF_USERNAME}")

In [ ]:
# ── Convert nanoGPT checkpoint → standalone HF-compatible package ──
import torch
import json
import pickle
import os
import sys

# Load the trained checkpoint
ckpt_path = 'out-tinystories/ckpt.pt'
assert os.path.exists(ckpt_path), f"Checkpoint not found at {ckpt_path}! Did training complete?"

checkpoint = torch.load(ckpt_path, map_location='cpu', weights_only=False)
model_args = checkpoint['model_args']
state_dict = checkpoint['config']

print("Model args from checkpoint:")
for k, v in model_args.items():
    print(f"  {k}: {v}")
print(f"\nBest val loss: {checkpoint.get('best_val_loss', 'N/A')}")
print(f"Training iter: {checkpoint.get('iter_num', 'N/A')}")

In [ ]:
# ── Reconstruct model and load weights ──
sys.path.insert(0, '.')
from model import GPTConfig, GPT

# Build config from checkpoint args
gptconf = GPTConfig(**model_args)
model = GPT(gptconf)

# nanoGPT saves with 'model.' prefix and may have unwanted keys
raw_sd = checkpoint['model']
unwanted_suffix = '._orig_mod.'
cleaned_sd = {}
for k, v in raw_sd.items():
    # Remove torch.compile prefix if present
    if unwanted_suffix in k:
        k = k.replace(unwanted_suffix, '.')
    cleaned_sd[k] = v

model.load_state_dict(cleaned_sd, strict=False)
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f"✅ Model loaded: {total_params:,} parameters")

In [ ]:
# ── Create a proper tokenizer package ──
import json, pickle, os

EXPORT_DIR = 'hf_export'
os.makedirs(EXPORT_DIR, exist_ok=True)

# Load meta
with open('data/tinystories/meta.pkl', 'rb') as f:
    meta = pickle.load(f)

itos = meta['itos']
stoi = meta['stoi']
vocab_size = meta['vocab_size']

# ── tokenizer_config.json ──
tokenizer_config = {
    "model_type": "char-level",
    "tokenizer_class": "PreTrainedTokenizerFast",
    "vocab_size": vocab_size,
    "model_max_length": model_args.get('block_size', 256),
    "padding_side": "right",
    "eos_token": "\n",
    "bos_token": "\n",
    "unk_token": "?"
}
with open(os.path.join(EXPORT_DIR, 'tokenizer_config.json'), 'w') as f:
    json.dump(tokenizer_config, f, indent=2)

# ── vocab.json ──
vocab_json = {ch: idx for ch, idx in stoi.items()}
with open(os.path.join(EXPORT_DIR, 'vocab.json'), 'w', encoding='utf-8') as f:
    json.dump(vocab_json, f, indent=2, ensure_ascii=False)

# ── special_tokens_map.json ──
special_tokens = {
    "eos_token": "\n",
    "bos_token": "\n",
    "unk_token": "?"
}
with open(os.path.join(EXPORT_DIR, 'special_tokens_map.json'), 'w') as f:
    json.dump(special_tokens, f, indent=2)

# ── char_tokenizer.py — standalone encode/decode helper ──
tokenizer_code = f'''"""Character-level tokenizer for nanoGPT TinyStories model."""
import json, os

_dir = os.path.dirname(os.path.abspath(__file__))
with open(os.path.join(_dir, "vocab.json"), "r", encoding="utf-8") as f:
    _vocab = json.load(f)
_ivocab = {{v: k for k, v in _vocab.items()}}

def encode(text: str) -> list[int]:
    return [_vocab.get(ch, _vocab.get("?", 0)) for ch in text]

def decode(ids: list[int]) -> str:
    return "".join(_ivocab.get(i, "?") for i in ids)

VOCAB_SIZE = {vocab_size}
'''
with open(os.path.join(EXPORT_DIR, 'char_tokenizer.py'), 'w') as f:
    f.write(tokenizer_code)

print(f"✅ Tokenizer package created in {EXPORT_DIR}/")
print(f"   vocab_size={vocab_size}, files: vocab.json, tokenizer_config.json, char_tokenizer.py")

In [ ]:
# ── Save model weights + config ──
import json

# Save bare PyTorch weights
torch.save(model.state_dict(), os.path.join(EXPORT_DIR, 'pytorch_model.bin'))

# Save model config
model_config = {
    "architectures": ["GPT"],
    "model_type": "nanogpt",
    "vocab_size": model_args['vocab_size'],
    "block_size": model_args['block_size'],
    "n_layer": model_args['n_layer'],
    "n_head": model_args['n_head'],
    "n_embd": model_args['n_embd'],
    "dropout": model_args.get('dropout', 0.0),
    "bias": model_args.get('bias', True),
    "torch_dtype": "float16",
    "total_params": total_params
}
with open(os.path.join(EXPORT_DIR, 'config.json'), 'w') as f:
    json.dump(model_config, f, indent=2)

# Also copy the nanoGPT model.py for easy re-use
import shutil
shutil.copy('model.py', os.path.join(EXPORT_DIR, 'model.py'))

print(f"✅ Model weights saved: {os.path.getsize(os.path.join(EXPORT_DIR, 'pytorch_model.bin')) / 1024:.0f} KB")

In [ ]:
# ── Generate a model card ──
model_card = f"""---
language: en
license: mit
tags:
  - nanogpt
  - text-generation
  - character-level
  - tinystories
  - pytorch
pipeline_tag: text-generation
---

# 🧠 nanoGPT — TinyStories Character-Level Model

A compact character-level GPT model trained on the [TinyStories](https://huggingface.co/datasets/roneneldan/TinyStories) dataset using [Karpathy's nanoGPT](https://github.com/karpathy/nanoGPT).

## Model Details

| Parameter | Value |
|-----------|-------|
| Parameters | {total_params:,} |
| Layers | {model_args['n_layer']} |
| Heads | {model_args['n_head']} |
| Embedding Dim | {model_args['n_embd']} |
| Context Length | {model_args['block_size']} |
| Vocab Size | {model_args['vocab_size']} (character-level) |
| Training Iters | 2000 |
| dtype | float16 |

## Usage

```python
import torch, json
from model import GPTConfig, GPT

# Load config
with open('config.json') as f:
    cfg = json.load(f)

# Build model
conf = GPTConfig(
    vocab_size=cfg['vocab_size'], block_size=cfg['block_size'],
    n_layer=cfg['n_layer'], n_head=cfg['n_head'], n_embd=cfg['n_embd'],
    dropout=cfg['dropout'], bias=cfg['bias']
)
model = GPT(conf)
model.load_state_dict(torch.load('pytorch_model.bin', map_location='cpu'))
model.eval()

# Tokenize & generate
from char_tokenizer import encode, decode
prompt = "Once upon a time"
ids = torch.tensor([encode(prompt)], dtype=torch.long)
out = model.generate(ids, max_new_tokens=200, temperature=0.8, top_k=40)
print(decode(out[0].tolist()))
```

## Training

Trained on Google Colab (T4 GPU) for ~10 minutes.
Dataset: First ~20 MB of TinyStories-V2-GPT4-train.

## License

MIT
"""

with open(os.path.join(EXPORT_DIR, 'README.md'), 'w') as f:
    f.write(model_card)

print("✅ Model card written")

In [ ]:
# ── Push everything to Hugging Face Hub ──
from huggingface_hub import HfApi, create_repo

REPO_NAME = "nanogpt-tinystories"
REPO_ID = f"{HF_USERNAME}/{REPO_NAME}"

# Create the repo (skip if already exists)
try:
    create_repo(REPO_ID, token=hf_token, repo_type='model', exist_ok=True)
    print(f"✅ Repository ready: https://huggingface.co/{REPO_ID}")
except Exception as e:
    print(f"Repo creation: {e}")

# Upload the entire export directory
api.upload_folder(
    folder_path=EXPORT_DIR,
    repo_id=REPO_ID,
    repo_type='model',
    token=hf_token,
    commit_message='Upload nanoGPT TinyStories model + char tokenizer'
)

print(f"\n🎉 Model uploaded! View at: https://huggingface.co/{REPO_ID}")

---
## 7. Deploy a Gradio Demo to Hugging Face Spaces 🚀
Creates an interactive text generation app on Hugging Face Spaces.

In [ ]:
import torch
import json
import gradio as gr
import os
import sys
import shutil
from huggingface_hub import HfApi, create_repo

# 1. Define Space Repo
SPACE_REPO = f"{HF_USERNAME}/{REPO_NAME}-demo"

# 2. Prepare Space Directory
SPACE_DIR = 'hf_space'
os.makedirs(SPACE_DIR, exist_ok=True)

# Copy necessary files from export
for fname in ['pytorch_model.bin', 'config.json', 'vocab.json', 'tokenizer_config.json', 'char_tokenizer.py', 'model.py']:
    src = os.path.join(EXPORT_DIR, fname)
    if os.path.exists(src):
        shutil.copy(src, os.path.join(SPACE_DIR, fname))

# Write requirements.txt
with open(os.path.join(SPACE_DIR, 'requirements.txt'), 'w') as f:
    f.write('torch\ngradio\n')

# 3. Create the fixed app.py (Removing allow_flagging)
app_code = f'''import torch
import json
import gradio as gr
from model import GPTConfig, GPT
from char_tokenizer import encode, decode

# Load model
with open("config.json") as f:
    cfg = json.load(f)

conf = GPTConfig(
    vocab_size=cfg["vocab_size"],
    block_size=cfg["block_size"],
    n_layer=cfg["n_layer"],
    n_head=cfg["n_head"],
    n_embd=cfg["n_embd"],
    dropout=0.0,
    bias=cfg.get("bias", True),
)
model = GPT(conf)
model.load_state_dict(torch.load("pytorch_model.bin", map_location="cpu"))
model.eval()

def generate_story(prompt, max_tokens, temperature, top_k):
    if not prompt: prompt = "Once upon a time"
    ids = torch.tensor([encode(prompt)], dtype=torch.long)
    out = model.generate(ids, max_new_tokens=int(max_tokens), temperature=float(temperature), top_k=int(top_k))
    return decode(out[0].tolist())

demo = gr.Interface(
    fn=generate_story,
    inputs=[
        gr.Textbox(label="Prompt", placeholder="Once upon a time", lines=2),
        gr.Slider(50, 500, value=200, step=10, label="Max Tokens"),
        gr.Slider(0.1, 2.0, value=0.8, step=0.1, label="Temperature"),
        gr.Slider(1, 100, value=40, step=1, label="Top-K"),
    ],
    outputs=gr.Textbox(label="Generated Story", lines=12),
    title="🧠 nanoGPT TinyStories",
)
demo.launch()
'''

with open(os.path.join(SPACE_DIR, 'app.py'), 'w') as f:
    f.write(app_code)

# 4. Push to Spaces
try:
    create_repo(SPACE_REPO, token=hf_token, repo_type='space', space_sdk='gradio', exist_ok=True)
    api.upload_folder(
        folder_path=SPACE_DIR,
        repo_id=SPACE_REPO,
        repo_type='space',
        token=hf_token,
        commit_message='Fix: remove allow_flagging for compatibility'
    )
    print(f"🎉 Fixed Space deployed! Visit: https://huggingface.co/spaces/{SPACE_REPO}")
except Exception as e:
    print(f"Error: {e}")

---
## ✅ Done!

**What was deployed:**
- 🤗 **Model**: `{HF_USERNAME}/nanogpt-tinystories` — weights, tokenizer, config, and model card
- 🚀 **Space**: `{HF_USERNAME}/nanogpt-tinystories-demo` — interactive Gradio demo

**Files uploaded per repo:**
| File | Description |
|------|-------------|
| `pytorch_model.bin` | Model weights |
| `config.json` | Model architecture config |
| `model.py` | nanoGPT model code |
| `vocab.json` | Character → ID mapping |
| `tokenizer_config.json` | Tokenizer metadata |
| `char_tokenizer.py` | Standalone encode/decode |
| `README.md` | Model card (model repo only) |
| `app.py` | Gradio app (Space only) |